In [7]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_PATH = Path("bbc-text.csv")

# Download bbc-text.csv if not present
if not DATA_PATH.exists():
    url = "https://raw.githubusercontent.com/mdsohaib/BBC-News-Classification/master/bbc-text.csv"
    print("Downloading:", url)
    df = pd.read_csv(url)
    df.to_csv(DATA_PATH, index=False)
else:
    df = pd.read_csv(DATA_PATH)

print(df.head())
print("rows:", len(df), "cols:", list(df.columns))
print(df["category"].value_counts())


        category                                               text
0           tech  tv future in the hands of viewers with home th...
1       business  worldcom boss  left books alone  former worldc...
2          sport  tigers wary of farrell  gamble  leicester say ...
3          sport  yeading face newcastle in fa cup premiership s...
4  entertainment  ocean s twelve raids box office ocean s twelve...
rows: 2225 cols: ['category', 'text']
category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


In [8]:
import re
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Basic cleaning

def clean_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^a-z\s]", " ", s)  # keep letters/spaces
    s = re.sub(r"\s+", " ", s).strip()
    return s

texts = df["text"].astype(str).map(clean_text).tolist()
labels = df["category"].astype(str).tolist()

# Encode labels to ints
label_to_id = {name: i for i, name in enumerate(sorted(set(labels)))}
id_to_label = {i: name for name, i in label_to_id.items()}
y = np.array([label_to_id[l] for l in labels], dtype=np.int32)
num_classes = len(label_to_id)

print("classes:", label_to_id)

# Train/test split (stratified)
from sklearn.model_selection import train_test_split

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Tokenize
VOCAB_SIZE = 20000
OOV_TOKEN = "<OOV>"

tok = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tok.fit_on_texts(X_train_txt)

train_seq = tok.texts_to_sequences(X_train_txt)
test_seq = tok.texts_to_sequences(X_test_txt)

# Pad
MAX_LEN = 250
train_pad = pad_sequences(train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
test_pad = pad_sequences(test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

print("train_pad:", train_pad.shape, "test_pad:", test_pad.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)


classes: {'business': 0, 'entertainment': 1, 'politics': 2, 'sport': 3, 'tech': 4}
train_pad: (1780, 250) test_pad: (445, 250)
y_train: (1780,) y_test: (445,)


In [9]:
from tensorflow.keras import layers

# LSTM model
EMBED_DIM = 128
LSTM_UNITS = 128
DROPOUT = 0.3

model = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),
    layers.SpatialDropout1D(DROPOUT),
    layers.LSTM(LSTM_UNITS, dropout=0.2, recurrent_dropout=0.0),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 250, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ (None, 250, 128)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,700,165 (10.30 MB)

 Trainable params: 2,700,165 (10.30 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Train
BATCH_SIZE = 64
EPOCHS = 15  # adjust to 10–20

callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.5, patience=2, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_pad,
    y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)


Epoch 1/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 105ms/step - accuracy: 0.2522 - loss: 1.5761 - val_accuracy: 0.3652 - val_loss: 1.5319 - learning_rate: 0.0010
Epoch 2/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - accuracy: 0.3227 - loss: 1.5155 - val_accuracy: 0.4101 - val_loss: 1.5101 - learning_rate: 0.0010
Epoch 3/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 95ms/step - accuracy: 0.4357 - loss: 1.4606 - val_accuracy: 0.4438 - val_loss: 1.4326 - learning_rate: 0.0010
Epoch 4/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 94ms/step - accuracy: 0.4557 - loss: 1.3113 - val_accuracy: 0.4213 - val_loss: 1.2208 - learning_rate: 0.0010
Epoch 5/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - accuracy: 0.4713 - loss: 1.2163 - val_accuracy: 0.4944 - val_loss: 1.2277 - learning_rate: 0.0010
Epoch 6/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 94ms/step - accuracy: 0.4950 - loss: 1.1144 - val_accuracy: 0.5056 - val_loss: 1.1969 - learning_rate: 0.0010
Epoch 7/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step - accuracy: 0.5768 - loss: 0.9944 - val_ac

In [11]:
# Evaluate: loss/accuracy
loss, acc = model.evaluate(test_pad, y_test, verbose=0)
print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {acc*100:.2f}%")

# Confusion matrix + Precision/Recall/F1
from sklearn.metrics import confusion_matrix, classification_report

y_prob = model.predict(test_pad, batch_size=256, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

a = confusion_matrix(y_test, y_pred)
print("Confusion matrix (rows=true, cols=pred):\n", a)

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=[id_to_label[i] for i in range(num_classes)], digits=4))


Test loss: 0.5504
Test accuracy: 84.27%
Confusion matrix (rows=true, cols=pred):
 [[ 90   5   4   1   2]
 [  6  60   7   4   0]
 [  3  22  56   1   2]
 [  1   0   0 101   0]
 [  1   2   8   1  68]]

Classification report:
               precision    recall  f1-score   support

     business     0.8911    0.8824    0.8867       102
entertainment     0.6742    0.7792    0.7229        77
     politics     0.7467    0.6667    0.7044        84
        sport     0.9352    0.9902    0.9619       102
         tech     0.9444    0.8500    0.8947        80

     accuracy                         0.8427       445
    macro avg     0.8383    0.8337    0.8341       445
 weighted avg     0.8460    0.8427    0.8426       445



## Compare with paper (Ezen‑Can, Springer 2023)

Paper target accuracy: **97.0%**

After training, fill in your achieved accuracy from the evaluation cell:

- Paper: **97.0%**
- Yours: **(acc * 100)%**

### Why your accuracy may differ

Common reasons your result can be below 97%:

- **Train/test split differences**: paper may use a specific split or cross‑validation; you’re using an 80/20 stratified split.
- **Preprocessing differences**: tokenization, max length, vocab size, stopword removal, etc.
- **Hyperparameters**: embedding dim, LSTM units, dropout, batch size, LR schedule.
- **Model details**: biLSTM, stacked LSTM, attention, pre-trained embeddings can improve results.
- **Randomness**: different seeds and library versions shift results slightly.


In [12]:
# Improved pipeline: TextVectorization + BiLSTM (typically much higher accuracy on BBC)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)

MAX_TOKENS = 30000
SEQ_LEN = 400

vectorize = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQ_LEN,
    standardize="lower_and_strip_punctuation",
)

# Adapt on training text only
vectorize.adapt(tf.data.Dataset.from_tensor_slices(X_train_txt).batch(64))

BATCH_SIZE = 64

def make_ds(texts, labels, training: bool):
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(texts), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).map(lambda t, y: (vectorize(t), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(X_train_txt, y_train, training=True)
test_ds = make_ds(X_test_txt, y_test, training=False)

# Keep a validation split from training
val_frac = 0.1
val_size = int(len(X_train_txt) * val_frac)

val_ds = make_ds(X_train_txt[:val_size], y_train[:val_size], training=False)
train_ds2 = make_ds(X_train_txt[val_size:], y_train[val_size:], training=True)

print("train/val/test sizes:", len(X_train_txt[val_size:]), val_size, len(X_test_txt))


train/val/test sizes: 1602 178 445


In [13]:
# Improved BiLSTM model
EMBED_DIM = 128

model2 = keras.Sequential([
    layers.Input(shape=(SEQ_LEN,)),
    layers.Embedding(input_dim=MAX_TOKENS, output_dim=EMBED_DIM),
    layers.Bidirectional(layers.LSTM(128, return_sequences=True)),
    layers.GlobalMaxPooling1D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax"),
])

model2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model2.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 400, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 400, 256)       │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 256)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,136,709 (15.78 MB)

 Trainable params: 4,136,709 (15.78 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# Train improved model
EPOCHS = 15  # 10–20

callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.5, patience=2, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True, verbose=1),
]

history2 = model2.fit(
    train_ds2,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)


Epoch 1/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 14s 510ms/step - accuracy: 0.2628 - loss: 1.5689 - val_accuracy: 0.4663 - val_loss: 1.4972 - learning_rate: 0.0010
Epoch 2/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 12s 456ms/step - accuracy: 0.4357 - loss: 1.3647 - val_accuracy: 0.4438 - val_loss: 1.1573 - learning_rate: 0.0010
Epoch 3/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 13s 486ms/step - accuracy: 0.6298 - loss: 0.9107 - val_accuracy: 0.8820 - val_loss: 0.5755 - learning_rate: 0.0010
Epoch 4/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 13s 502ms/step - accuracy: 0.8727 - loss: 0.4295 - val_accuracy: 0.8933 - val_loss: 0.3385 - learning_rate: 0.0010
Epoch 5/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 15s 579ms/step - accuracy: 0.9544 - loss: 0.1787 - val_accuracy: 0.9438 - val_loss: 0.1390 - learning_rate: 0.0010
Epoch 6/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 15s 575ms/step - accuracy: 0.9894 - loss: 0.0567 - val_accuracy: 0.9607 - val_loss: 0.1225 - learning_rate: 0.0010
Epoch 7/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 15s 569ms/step - accuracy: 0.9956 - loss: 0.

In [15]:
# Evaluate improved model
loss2, acc2 = model2.evaluate(test_ds, verbose=0)
print(f"Improved Test loss: {loss2:.4f}")
print(f"Improved Test accuracy: {acc2*100:.2f}%")

# Confusion matrix + Precision/Recall/F1
from sklearn.metrics import confusion_matrix, classification_report

y_prob2 = model2.predict(test_ds, verbose=0)
y_pred2 = np.argmax(y_prob2, axis=1)

cm2 = confusion_matrix(y_test, y_pred2)
print("Confusion matrix (rows=true, cols=pred):\n", cm2)

print("\nClassification report:")
print(classification_report(y_test, y_pred2, target_names=[id_to_label[i] for i in range(num_classes)], digits=4))


Improved Test loss: 0.1238
Improved Test accuracy: 95.73%
Confusion matrix (rows=true, cols=pred):
 [[ 95   3   4   0   0]
 [  0  75   1   1   0]
 [  1   3  78   0   2]
 [  0   1   0 101   0]
 [  1   2   0   0  77]]

Classification report:
               precision    recall  f1-score   support

     business     0.9794    0.9314    0.9548       102
entertainment     0.8929    0.9740    0.9317        77
     politics     0.9398    0.9286    0.9341        84
        sport     0.9902    0.9902    0.9902       102
         tech     0.9747    0.9625    0.9686        80

     accuracy                         0.9573       445
    macro avg     0.9554    0.9573    0.9559       445
 weighted avg     0.9586    0.9573    0.9575       445

